# Proceso de validacion de pagos y prestamos

Proposito del script: 
- Verificar fecha_ultimo_pago_real de la tabla prestamos coincida con la fecha del ultimo pago registrado en la tabla pagos.
- Verificar que saldo_capital_vigente de prestamos, coincida al saldo_capital_despues_pago del ultimo pago realizado en la tabla pagos. 
- Verificar que los dias_mora de prestamos, coincida con dias_retraso del ultimo pago realizado en la tabla pagos.
- Verificar que clasificacion_riesgo_sbs, sea la correcta para los dias de mora para dicho prestamo. 
- Modificar aquellos registros que incumplan la logica del negocio

# Cargando los Archivos Limpios

In [1]:
import pandas as pd 
from funciones import formato_clasificacion_riesgo_sbs,asignacion_estado_prestamo
from conexiones_y_rutas import obtener_ruta_archivo

df_pagos = pd.read_parquet(obtener_ruta_archivo("archivos_limpios","limpio_pagos.parquet"))
df_pagos_tra = df_pagos.copy()

df_prestamos = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_prestamos.parquet"))
df_prestamos_tra = df_prestamos.copy()

# Proceso de Limpieza 

Como partimos de datos que en su payoria ya estan limpios, solo me voy a centrar en las columnas que necesitan cambios en prestamos.

In [2]:
df_pagos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134911 entries, 0 to 134910
Data columns (total 17 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   pago_id                     134911 non-null  int64         
 1   prestamo_id                 134911 non-null  int64         
 2   numero_cuota                134911 non-null  int64         
 3   fecha_vencimiento_cuota     134911 non-null  datetime64[ns]
 4   fecha_pago                  134911 non-null  datetime64[ns]
 5   monto_cuota_programada      134911 non-null  float64       
 6   monto_capital_programado    134911 non-null  float64       
 7   monto_interes_programado    134911 non-null  float64       
 8   monto_pagado_total          134911 non-null  float64       
 9   monto_capital_pagado        134911 non-null  float64       
 10  monto_interes_pagado        134911 non-null  float64       
 11  monto_mora_pagado           134911 non-

In [3]:
df_prestamos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6495 entries, 0 to 6494
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   prestamo_id                   6495 non-null   int64         
 1   cliente_id                    6495 non-null   int64         
 2   sucursal_id                   6495 non-null   int64         
 3   producto_id                   6495 non-null   int64         
 4   oficial_id                    6495 non-null   int64         
 5   numero_contrato               6495 non-null   object        
 6   fecha_otorgamiento            6495 non-null   datetime64[ns]
 7   fecha_vencimiento             6495 non-null   datetime64[ns]
 8   monto_original                6495 non-null   float64       
 9   saldo_capital_vigente         6495 non-null   float64       
 10  tasa_interes_nominal_anual    6495 non-null   float64       
 11  tasa_interes_efectiva_anual   

## Obtenemos la informacion del ultimo pago realizado

In [4]:
df_prestamos_cuotas = df_pagos_tra[['prestamo_id','numero_cuota']].copy()
df_info_prestamos = df_prestamos_tra[['prestamo_id','numero_cuotas_total','clasificacion_riesgo_sbs','fecha_ultimo_pago_real','dias_mora','saldo_capital_vigente','numero_cuotas_total']].copy()

# Calcula el indice de la ultima cuota  
df_ultimo_pago = df_prestamos_cuotas.groupby('prestamo_id').agg(
    idx_cuota_maxima = ('numero_cuota','idxmax'),
).reset_index()
# Encuentra los valores asociados a estos indices 
df_ultimo_pago_por_prestamo = df_pagos_tra.loc[df_ultimo_pago.idx_cuota_maxima,['pago_id','prestamo_id','dias_retraso','numero_cuota','fecha_pago','saldo_capital_despues_pago']].copy()
# LEFT JOIN, prestamo_id
df_merge_ultimo_pago = df_info_prestamos.merge(
    right = df_ultimo_pago_por_prestamo,
    on='prestamo_id',
    how='left'
)

## fecha_ultimo_pago_real

**Nota**: Aqui ya no verifico si existen fechas de pago futuras, o la relacion entre las demas fechas como, fecha_apertura de sucursal, fecha_ingreso de oficial o fecha registro de clientes aqui solo verifico que las fechas sean correctas.

In [5]:
# Muestras los registrso donde no coincidan las ultimas fechas 
# Resultados Esperados: Tabla Vacia
df_merge_ultimo_pago[df_merge_ultimo_pago.fecha_pago != df_merge_ultimo_pago.fecha_ultimo_pago_real]

,prestamo_id,numero_cuotas_total,clasificacion_riesgo_sbs,fecha_ultimo_pago_real,dias_mora,saldo_capital_vigente,numero_cuotas_total,pago_id,dias_retraso,numero_cuota,fecha_pago,saldo_capital_despues_pago
3,4,6,Normal,2024-10-16,0,0.00,6,86,16,6,2024-11-01,0.00
8,9,48,Normal,2024-01-30,0,0.00,48,238,93,48,2024-05-02,0.00
13,14,18,Normal,2023-06-26,0,0.00,18,363,73,18,2023-09-07,0.00
17,18,48,Pérdida,2024-12-07,152,20922.61,48,469,6,38,2024-12-13,12908.39
24,25,30,Normal,2024-12-21,0,4731.62,30,578,1,4,2024-12-22,4731.63
...,...,...,...,...,...,...,...,...,...,...,...,...
6475,6480,9,Pérdida,2024-06-28,360,1999.80,9,134677,36,9,2024-08-03,0.00
6481,6486,18,Normal,2023-04-29,0,0.00,18,134757,47,18,2023-06-15,0.00
6482,6487,9,Normal,2022-11-04,0,0.00,9,134766,30,9,2022-12-04,0.00
6486,6491,24,Normal,2023-01-29,0,0.00,24,134856,24,24,2023-02-22,0.00


In [6]:
# Reemplaza y verifica valores 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra['fecha_ultimo_pago_real'] = df_merge_ultimo_pago.fecha_pago
df_prestamos_tra[df_prestamos_tra.fecha_ultimo_pago_real != df_merge_ultimo_pago.fecha_pago]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## saldo_capital_vigente

In [7]:
# Muestra los registros donde no coindan los saldos 
# Resultados Esperados: Tabla Vacia
df_merge_ultimo_pago[df_merge_ultimo_pago.saldo_capital_despues_pago != df_merge_ultimo_pago.saldo_capital_vigente]

,prestamo_id,numero_cuotas_total,clasificacion_riesgo_sbs,fecha_ultimo_pago_real,dias_mora,saldo_capital_vigente,numero_cuotas_total,pago_id,dias_retraso,numero_cuota,fecha_pago,saldo_capital_despues_pago
0,1,30,Normal,2024-12-27,0,3596.11,30,20,0,20,2024-12-27,3596.20
1,2,54,Normal,2024-12-23,0,1473.65,54,67,0,47,2024-12-23,1473.62
2,3,48,Normal,2024-12-16,0,10640.44,48,80,0,13,2024-12-16,10640.43
5,6,306,Normal,2024-12-14,0,188558.10,306,169,0,53,2024-12-14,188558.01
9,10,54,Pérdida,2024-12-28,252,32134.06,54,281,0,43,2024-12-28,18910.91
...,...,...,...,...,...,...,...,...,...,...,...,...
6488,6493,360,Normal,2024-12-31,0,376984.52,360,134892,0,27,2024-12-31,376984.64
6490,6495,288,Normal,2024-12-11,0,76207.15,288,134911,0,16,2024-12-11,76207.19
6492,872,27,Pérdida,2024-02-08,354,1234.00,27,17873,0,27,2024-02-08,0.00
6493,202,21,Normal,2024-12-20,0,1954.81,21,4155,0,16,2024-12-20,1954.89


In [8]:
# Reemplaza y verifica valores 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra['saldo_capital_vigente'] = df_merge_ultimo_pago.saldo_capital_despues_pago
df_prestamos_tra[df_prestamos_tra.saldo_capital_vigente != df_merge_ultimo_pago.saldo_capital_despues_pago]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## dias_mora

In [9]:
# Muestra los registros donde no coindan los dias de mora del ultimos pago 
# Resultados Esperados: Tabla Vacia
df_merge_ultimo_pago[df_merge_ultimo_pago.dias_retraso != df_merge_ultimo_pago.dias_mora]

,prestamo_id,numero_cuotas_total,clasificacion_riesgo_sbs,fecha_ultimo_pago_real,dias_mora,saldo_capital_vigente,numero_cuotas_total,pago_id,dias_retraso,numero_cuota,fecha_pago,saldo_capital_despues_pago
3,4,6,Normal,2024-10-16,0,0.00,6,86,16,6,2024-11-01,0.00
8,9,48,Normal,2024-01-30,0,0.00,48,238,93,48,2024-05-02,0.00
9,10,54,Pérdida,2024-12-28,252,32134.06,54,281,0,43,2024-12-28,18910.91
12,13,12,Deficiente,2024-03-28,52,3624.84,12,345,0,12,2024-03-28,0.00
13,14,18,Normal,2023-06-26,0,0.00,18,363,73,18,2023-09-07,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
6482,6487,9,Normal,2022-11-04,0,0.00,9,134766,30,9,2022-12-04,0.00
6485,6490,30,Pérdida,2024-01-01,340,2395.83,30,134832,0,30,2024-01-01,0.00
6486,6491,24,Normal,2023-01-29,0,0.00,24,134856,24,24,2023-02-22,0.00
6489,6494,3,Normal,2020-04-06,0,0.00,3,134895,69,3,2020-06-14,0.00


In [10]:
# Reemplaza y verifica valores 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra['dias_mora'] = df_merge_ultimo_pago.dias_retraso
df_prestamos_tra[df_prestamos_tra.dias_mora != df_merge_ultimo_pago.dias_retraso]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## clasificacion_riesgo_sbs

In [11]:
df_prestamos_tra["clasificacion_riesgo_sbs"] = df_prestamos_tra.apply(
    axis = 1,
    func = lambda x:
    formato_clasificacion_riesgo_sbs(x['tipo_credito'],x['dias_mora'])
)
df_prestamos_tra.clasificacion_riesgo_sbs.unique()

array(['Normal', 'CPP', 'Dudoso', 'Deficiente', 'Pérdida'], dtype=object)

## estado

In [12]:
df_prestamos_tra["estado"] = df_prestamos_tra.apply(
    axis = 1,
    func = lambda x: asignacion_estado_prestamo(x["numero_cuotas_pendientes"],x["dias_mora"],x["estado"],x["tipo_credito"])
)

# Carga los Nuevos Registros y Sobrescribe el archivo original 

In [13]:
df_prestamos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6495 entries, 0 to 6494
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   prestamo_id                   6495 non-null   int64         
 1   cliente_id                    6495 non-null   int64         
 2   sucursal_id                   6495 non-null   int64         
 3   producto_id                   6495 non-null   int64         
 4   oficial_id                    6495 non-null   int64         
 5   numero_contrato               6495 non-null   object        
 6   fecha_otorgamiento            6495 non-null   datetime64[ns]
 7   fecha_vencimiento             6495 non-null   datetime64[ns]
 8   monto_original                6495 non-null   float64       
 9   saldo_capital_vigente         6495 non-null   float64       
 10  tasa_interes_nominal_anual    6495 non-null   float64       
 11  tasa_interes_efectiva_anual   

In [14]:
df_prestamos_tra.head()

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
0,1,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,3596.20,...,10,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27
1,2,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,1473.62,...,7,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23
2,3,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,10640.43,...,35,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16
3,4,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,0.00,...,0,Cancelado,16,CPP,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-11-01
4,5,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06


In [15]:
df_prestamos_tra.to_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_prestamos.parquet"),
    index=False
)